# Interseção SICOR Cerrado × Embargos do Ibama por ID_GLEBA2

Este notebook cruza o arquivo `sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv` com o shapefile `adm_embargo_ibama_a.shp`.

## Regra central

- `ID_GLEBA2` é a referência espacial;
- cada `ID_GLEBA2` distinto é submetido à interseção somente uma vez;
- se o ID reaparecer, o notebook valida se a geometria é a mesma e não repete o cálculo;
- o resultado espacial é depois associado a todas as linhas originais que compartilham aquele ID.

## Produtos

1. resultado único por `ID_GLEBA2 × feição de embargo`;
2. resultado expandido para todos os vínculos originais do SICOR;
3. resumo de todos os `ID_GLEBA2`, inclusive os sem interseção.

O cálculo usa `EPSG:5880` para áreas, remove contatos apenas de borda e preserva os atributos do embargo com o prefixo `EMB_`.


In [ ]:
# ============================================================
# 1. INSTALAR, IMPORTAR E MONTAR O GOOGLE DRIVE
# ============================================================

%pip install -q --upgrade geopandas pyogrio shapely

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely import force_2d, intersection as shapely_intersection, make_valid, set_precision
from shapely.errors import GEOSException

warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 180)

print('Ambiente preparado.')


Mounted at /content/drive
Ambiente preparado.


In [ ]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

ARQUIVO_SICOR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/"
    "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
)

ARQUIVO_EMBARGO = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Base/adm_embargo_ibama_a.shp"
)

# Pasta nova para não sobrescrever resultados anteriores.
PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/intersecoes/embargos/"
    "resultados_por_id_gleba2"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_INTERSECAO_POR_ID = (
    PASTA_SAIDA / "intersecao_embargos_unica_por_id_gleba2.csv"
)

ARQUIVO_INTERSECAO_EXPANDIDA = (
    PASTA_SAIDA / "intersecao_sicor_embargos_expandida_por_id_gleba2.csv"
)

ARQUIVO_RESUMO_POR_ID = (
    PASTA_SAIDA / "resumo_embargos_por_id_gleba2.csv"
)

CRS_SICOR = "EPSG:4326"
CRS_AREA = "EPSG:5880"
TAMANHO_BLOCO = 20_000
SEPARADOR_SAIDA = ";"

for caminho in [ARQUIVO_SICOR, ARQUIVO_EMBARGO]:
    print(f"{'OK' if caminho.exists() else 'NÃO ENCONTRADO'} | {caminho}")

if not ARQUIVO_SICOR.exists():
    raise FileNotFoundError(f"CSV do SICOR não encontrado: {ARQUIVO_SICOR}")

if not ARQUIVO_EMBARGO.exists():
    raise FileNotFoundError(f"Shapefile de embargos não encontrado: {ARQUIVO_EMBARGO}")

for extensao in [".dbf", ".shx", ".prj"]:
    componente = ARQUIVO_EMBARGO.with_suffix(extensao)
    print(f"{'OK' if componente.exists() else 'ATENÇÃO'} | {componente.name}")

print("\nPasta de saída:")
print(PASTA_SAIDA)


OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/adm_embargo_ibama_a.shp
OK | adm_embargo_ibama_a.dbf
OK | adm_embargo_ibama_a.shx
OK | adm_embargo_ibama_a.prj

Pasta de saída:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/embargos/resultados_por_id_gleba2


In [ ]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

CODIFICACOES = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
SEPARADORES = [';', ',', '\t', '|']


def remover_acentos(texto):
    return ''.join(
        caractere
        for caractere in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip().replace('#', '')
    nome = re.sub(r'[^A-Z0-9]+', '_', nome)
    return re.sub(r'_+', '_', nome).strip('_')


def normalizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f'A normalização produziu colunas duplicadas: {duplicadas}')

    return df


def detectar_csv(caminho):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    dtype=str,
                    low_memory=False,
                )

                if teste.shape[1] <= 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        'Não foi possível detectar a estrutura do CSV. '
        f'Último erro: {ultimo_erro}'
    )


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    for candidato in [normalizar_nome_coluna(c) for c in candidatos]:
        if candidato in colunas:
            return candidato

    if obrigatoria:
        raise KeyError(
            'Nenhuma das colunas esperadas foi encontrada: '
            f'{[normalizar_nome_coluna(c) for c in candidatos]}'
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lstrip('\ufeff')
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, '<NA>': pd.NA})
    )


def corrigir_geometrias(gdf):
    quantidade_entrada = len(gdf)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    mascara_invalida = ~gdf.geometry.is_valid
    quantidade_invalidas = int(mascara_invalida.sum())

    if quantidade_invalidas:
        try:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].make_valid()
            )
        except Exception:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].buffer(0)
            )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    quantidade_descartadas = quantidade_entrada - len(gdf)
    return gdf, quantidade_invalidas, quantidade_descartadas


def serie_wkt_para_geometria(serie):
    return gpd.GeoSeries.from_wkt(
        serie.astype('string'),
        crs=CRS_SICOR,
        on_invalid='ignore',
    )


def preparar_geometria_topologica(geometria, precisao=None):
    if geometria is None:
        return None

    try:
        if geometria.is_empty:
            return None
        geometria = force_2d(geometria)

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        if precisao is not None:
            geometria = set_precision(
                geometria,
                grid_size=precisao,
                mode='valid_output',
            )

        if geometria is None or geometria.is_empty:
            return None

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        return geometria

    except Exception:
        try:
            geometria = geometria.buffer(0)
            return None if geometria.is_empty else geometria
        except Exception:
            return None


def calcular_intersecoes_seguras(geometrias_esquerda, geometrias_direita, crs):
    resultados = []
    quantidade_recuperadas = 0
    quantidade_falhas = 0

    for geometria_esquerda, geometria_direita in zip(
        geometrias_esquerda.to_numpy(),
        geometrias_direita.to_numpy(),
    ):
        esquerda = preparar_geometria_topologica(geometria_esquerda, 0.01)
        direita = preparar_geometria_topologica(geometria_direita, 0.01)

        if esquerda is None or direita is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        intersecao = None
        recuperada = False

        try:
            intersecao = shapely_intersection(esquerda, direita, grid_size=0.01)
        except (GEOSException, ValueError, TypeError):
            recuperada = True

            try:
                esquerda_10 = preparar_geometria_topologica(esquerda, 0.10)
                direita_10 = preparar_geometria_topologica(direita, 0.10)

                if esquerda_10 is not None and direita_10 is not None:
                    intersecao = shapely_intersection(
                        esquerda_10,
                        direita_10,
                        grid_size=0.10,
                    )
            except (GEOSException, ValueError, TypeError):
                intersecao = None

            if intersecao is None:
                try:
                    intersecao = esquerda.buffer(0).intersection(direita.buffer(0))
                except Exception:
                    intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        try:
            if not intersecao.is_empty and not intersecao.is_valid:
                intersecao = make_valid(intersecao)
        except Exception:
            try:
                intersecao = intersecao.buffer(0)
            except Exception:
                intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
        else:
            resultados.append(intersecao)
            if recuperada:
                quantidade_recuperadas += 1

    return (
        gpd.GeoSeries(resultados, index=pd.RangeIndex(len(resultados)), crs=crs),
        quantidade_recuperadas,
        quantidade_falhas,
    )


In [ ]:
# ============================================================
# 4. LER E PREPARAR OS EMBARGOS DO IBAMA
# ============================================================

embargos = gpd.read_file(ARQUIVO_EMBARGO, engine='pyogrio')
crs_original_embargo = embargos.crs
coluna_geometria_original = embargos.geometry.name

if crs_original_embargo is None:
    raise ValueError(
        'O adm_embargo_ibama_a.shp não possui CRS. Verifique se o arquivo .prj '
        'está na mesma pasta do shapefile.'
    )

embargos = normalizar_colunas(embargos)
coluna_geometria_normalizada = normalizar_nome_coluna(coluna_geometria_original)

colunas_geometricas = [
    coluna
    for coluna in embargos.columns
    if isinstance(embargos[coluna].dtype, gpd.array.GeometryDtype)
]

if coluna_geometria_normalizada not in embargos.columns:
    if not colunas_geometricas:
        raise KeyError('Nenhuma coluna geométrica foi localizada no shapefile de embargos.')
    coluna_geometria_normalizada = colunas_geometricas[0]

embargos = gpd.GeoDataFrame(
    embargos,
    geometry=coluna_geometria_normalizada,
    crs=crs_original_embargo,
)

if embargos.geometry.name != 'geometry':
    embargos = embargos.rename_geometry('geometry')

quantidade_embargo_original = len(embargos)
embargos, invalidas_embargo, descartadas_embargo = corrigir_geometrias(embargos)
embargos = embargos.reset_index(drop=True)
embargos['EMB_ID_INTERNO'] = np.arange(1, len(embargos) + 1)

renomear_embargo = {
    coluna: f'EMB_{coluna}'
    for coluna in embargos.columns
    if coluna not in {'geometry', 'EMB_ID_INTERNO'}
}
embargos = embargos.rename(columns=renomear_embargo).set_geometry('geometry')
embargos = embargos.to_crs(CRS_SICOR).reset_index(drop=True)

embargos_area = embargos.to_crs(CRS_AREA)
embargos['EMB_AREA_TOTAL_HA'] = embargos_area.geometry.area / 10_000

COLUNAS_EMB_SAIDA = [coluna for coluna in embargos.columns if coluna != 'geometry']

print('Feições originais de embargos:', f'{quantidade_embargo_original:,}')
print('Geometrias inválidas corrigidas:', f'{invalidas_embargo:,}')
print('Geometrias descartadas:', f'{descartadas_embargo:,}')
print('Feições válidas finais:', f'{len(embargos):,}')
print('CRS espacial:', embargos.crs)
print('CRS para áreas:', CRS_AREA)

display(embargos.head())


Feições originais de embargos: 88,586
Geometrias inválidas corrigidas: 45
Geometrias descartadas: 13,910
Feições válidas finais: 74,676
CRS espacial: EPSG:4326
CRS para áreas: EPSG:5880


,EMB_OBJECTID,EMB_SEQ_TAD,EMB_NUM_TAD,EMB_SERIE_TAD,EMB_OPERACAO,EMB_ORIGEM_GEO,EMB_COD_UF,EMB_UF,EMB_COD_MUNICI,EMB_MUNICIPIO,EMB_NOME_IMOVE,EMB_DES_LOCALI,EMB_NOME_EMBAR,EMB_CPF_CNPJ_E,EMB_SIT_DESMAT,EMB_TIPO_AREA,EMB_NUM_AUTO_I,EMB_SERIE_AUTO,EMB_COD_TIPO_B,EMB_DES_TIPO_B,EMB_UNID_CONTR,EMB_ORDEM_FISC,EMB_CD_ACAO_FI,EMB_NUM_PROCES,EMB_DES_TAD,EMB_DES_INFRAC,EMB_NUM_LONGIT,EMB_NUM_LATITU,EMB_DAT_EMBARG,EMB_DAT_IMPRES,EMB_DAT_ULT_AL,EMB_NUM_LONG00,EMB_NUM_LATI00,EMB_QTD_AREA_D,EMB_QTD_AREA_E,EMB_DAT_ULT_00,EMB_ST_AREA_SH,EMB_ST_PERIMET,geometry,EMB_ID_INTERNO,EMB_AREA_TOTAL_HA
0,25298,1639007,756611,E,None,Polígono,13,AM,1302405,Lábrea,None,Coordenadas de Referencia 09°21'43'' S - 66°35'05'' W,DESCONHECIDO,None,D,Desmatamento,None,None,None,None,DIFIS - Manaus/AM,DF590151,None,02024102709201736,"Artigo 78 da Lei 5.172/66 § unico , Artigo 50 da lei federal 9.605/98, Artigo 16,101,108 § 2° do decreto federal 6.514/08, artigo 51 da lei federal 12.651/12, fica embargado po...",None,66° 35' 04.999'' W,09° 21' 42.998'' S,11/07/17 14:48:00,11/07/17 14:53:30,28/01/19 13:36:23,-66.584724,-9.361944,NaN,235.268295,28/01/19 13:36:23,0.000194,0.081991,"POLYGON ((-66.58977 -9.36237, -66.59222 -9.36302, -66.59345 -9.35825, -66.59328 -9.3527, -66.59038 -9.35359, -66.58827 -9.35534, -66.58682 -9.35405, -66.58549 -9.34987, -66.584...",1,240.829352
1,25299,1472789,602347,E,None,Polígono,29,BA,2927705,Santa Cruz Cabrália,None,Sítio Santa Bárbara,WELLINGTON RODRIGUES DANTAS,04449165551,N,Outros,9168921,E,1,Mata Atlantica,Gerência Executiva do Ibama em Eunápolis/BA,None,None,02059100167201786,"Fica embargado o uso e as atividades que possam dificultar ou impedir a regeneração natural de uma área de 11,5 Hectares delimitada pelas Coordenadas geográficas apresentadas n...",Infração da Flora(Não Classificada-Móvel),39° 10' 52.000'' W,16° 10' 41.999'' S,11/07/17 16:49:00,11/07/17 17:08:59,03/10/17 16:34:45,-39.181110,-16.178333,3.800000,11.508800,03/10/17 16:34:45,0.000010,0.013697,"POLYGON ((-39.18127 -16.17836, -39.18262 -16.18181, -39.18262 -16.18182, -39.18321 -16.18192, -39.18354 -16.1822, -39.18446 -16.18167, -39.18521 -16.18124, -39.18382 -16.17873,...",2,11.865803
2,25300,1610984,729878,E,None,Polígono,15,PA,1500602,Altamira,None,"Estrada Vicinal Celeste, km 180, Altamira PA.",EMILIO CARLO NOGUEIRA BATAGIN,08972866857,N,None,9126094,E,4,Amazonia,Unidade Técnica Nível 1 em Altamira/PA,DF590154,None,02048101405201708,"Fica embargadas todas as atividades produtivas, excetuando-se as de proteção ambientais e as determinadas e ou autorizadas pelas autoridades competentes, na area objeto do AI n...",Infração da Flora(Não Classificada-Móvel),54° 51' 47.002'' W,06° 28' 54.001'' S,12/07/17 08:40:00,12/07/17 16:16:08,17/07/17 12:27:36,-54.863056,-6.481667,NaN,413.235504,17/07/17 12:27:36,0.000338,0.096284,"POLYGON ((-54.86299 -6.50415, -54.87181 -6.47247, -54.86902 -6.47099, -54.87005 -6.46765, -54.86555 -6.46752, -54.86298 -6.46842, -54.85874 -6.47125, -54.85887 -6.47215, -54.86...",3,413.272987
3,25301,1629339,746952,E,None,Polígono,31,MG,3139300,Manga,None,"fazenda agropasto, estrada São João das Missões/Manga, município de Manga MG, coordenada da entrada 14°52'05,29''-S 44°04'37,98''-W.",REGINALDO FRAGA GUEDES,03493521600,D,Desmatamento,9137930,E,5,Caatinga,DIFIS - Belo Horizonte/MG,MG048208,None,02566100124201771,"Lei Federal 9605/98 70 1° 72 II,VII Decreto Federal 6514/08 3 II,VII 50 1° Lei 11.428/2006 2°. Fica embargada a área de 28,6386 hectares, objeto do AI n° 9137930-E nas coord...",Infração da Flora(Não Classificada-Móvel),44° 03' 59.000'' W,14° 52' 05.002'' S,12/07/17 10:58:00,12/07/17 11:26:27,08/08/17 08:46:25,-44.066387,-14.868056,28.639000,28.641100,08/08/17 08:46:25,0.000024,0.024910,"POLYGON ((-44.0653 -14.86637, -44.06267 -14.86568, -44.06285 -14.8666, -44.06295 -14.86689, -44.06303 -14.86889, -44.06449 -14.86971, -44.06449 -14.86997, -44.06475 -14.86997, ...",4,29.045715
4,25302,1609835,728729,E,None,Polígono,15,PA,1500602,Altamira,

In [ ]:
# ============================================================
# 5. DETECTAR E VALIDAR AS COLUNAS DO SICOR
# ============================================================

encoding_sicor, separador_sicor, colunas_sicor = detectar_csv(ARQUIVO_SICOR)

COLUNA_ID_GLEBA2 = localizar_coluna(
    colunas_sicor,
    ['ID_GLEBA2', 'ID_GLEBA_2'],
)

COLUNA_GEOMETRIA_SICOR = localizar_coluna(
    colunas_sicor,
    ['GT_GEOMETRIA', 'GT_GEOMETRIA_REPRESENTATIVA', 'GEOMETRIA', 'GEOMETRY', 'WKT'],
)

COLUNA_REF_BACEN = localizar_coluna(
    colunas_sicor,
    ['REF_BACEN', '#REF_BACEN', 'CD_REF_BACEN'],
    obrigatoria=False,
)

print('Codificação:', encoding_sicor)
print('Separador:', repr(separador_sicor))
print('Coluna-chave:', COLUNA_ID_GLEBA2)
print('Coluna de geometria:', COLUNA_GEOMETRIA_SICOR)
print('Coluna REF_BACEN:', COLUNA_REF_BACEN)
print('Quantidade de colunas:', len(colunas_sicor))

if COLUNA_ID_GLEBA2 == COLUNA_GEOMETRIA_SICOR:
    raise RuntimeError('A coluna do ID e a coluna geométrica não podem ser a mesma.')


Codificação: utf-8-sig
Separador: ';'
Coluna-chave: ID_GLEBA2
Coluna de geometria: GT_GEOMETRIA
Coluna REF_BACEN: REF_BACEN
Quantidade de colunas: 27


In [ ]:
# ============================================================
# 6. INTERSECTAR UMA ÚNICA VEZ CADA ID_GLEBA2
# ============================================================

# Guarda somente uma assinatura numérica da geometria de cada ID.
# Isso permite reconhecer IDs já processados sem manter todos os WKT na memória.
assinatura_geometria_por_id = {}
todos_ids_encontrados = set()
ids_com_geometria_valida = set()
resultados_intersecao = []

total_linhas_lidas = 0
total_linhas_com_id_e_geometria = 0
total_ids_novos = 0
total_repeticoes_puladas = 0
total_ids_geometria_valida = 0
total_geometrias_invalidas_corrigidas = 0
total_geometrias_descartadas = 0
total_candidatos_espaciais = 0
total_intersecoes_validas = 0
total_intersecoes_recuperadas = 0
total_falhas_topologicas = 0
total_contatos_sem_area = 0

leitor = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    bloco = normalizar_colunas(bloco)
    total_linhas_lidas += len(bloco)

    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    todos_ids_encontrados.update(
        bloco[COLUNA_ID_GLEBA2].dropna().astype(str).tolist()
    )

    bloco[COLUNA_GEOMETRIA_SICOR] = (
        bloco[COLUNA_GEOMETRIA_SICOR].astype('string').str.strip()
    )

    bloco = bloco.dropna(
        subset=[COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
    ).copy()
    bloco = bloco[bloco[COLUNA_GEOMETRIA_SICOR].ne('')].copy()

    total_linhas_com_id_e_geometria += len(bloco)

    if bloco.empty:
        print(f'Bloco {numero_bloco:03d}: sem ID_GLEBA2 e geometria válidos')
        continue

    bloco['_HASH_GEOMETRIA'] = (
        pd.util.hash_pandas_object(
            bloco[COLUNA_GEOMETRIA_SICOR],
            index=False,
        )
        .astype('uint64')
    )

    # Validação dentro do próprio bloco.
    geometrias_por_id_no_bloco = bloco.groupby(
        COLUNA_ID_GLEBA2,
        dropna=False,
    )['_HASH_GEOMETRIA'].nunique()

    ids_inconsistentes = geometrias_por_id_no_bloco[
        geometrias_por_id_no_bloco > 1
    ].index.tolist()

    if ids_inconsistentes:
        raise ValueError(
            'Um mesmo ID_GLEBA2 foi associado a geometrias diferentes. '
            f'Primeiros IDs inconsistentes: {ids_inconsistentes[:10]}'
        )

    representantes = bloco.drop_duplicates(
        subset=[COLUNA_ID_GLEBA2],
        keep='first',
    ).reset_index(drop=True)

    mascara_id_novo = []

    # Validação também contra IDs que apareceram em blocos anteriores.
    for id_atual, hash_atual in representantes[
        [COLUNA_ID_GLEBA2, '_HASH_GEOMETRIA']
    ].itertuples(index=False, name=None):
        hash_atual = int(hash_atual)

        if id_atual in assinatura_geometria_por_id:
            if assinatura_geometria_por_id[id_atual] != hash_atual:
                raise ValueError(
                    f'O ID_GLEBA2 {id_atual} apareceu com geometrias diferentes '
                    'em blocos distintos do CSV.'
                )
            mascara_id_novo.append(False)
        else:
            assinatura_geometria_por_id[id_atual] = hash_atual
            mascara_id_novo.append(True)

    representantes_novos = representantes.loc[
        mascara_id_novo
    ].copy().reset_index(drop=True)

    total_ids_novos += len(representantes_novos)
    total_repeticoes_puladas += len(bloco) - len(representantes_novos)

    if representantes_novos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            'todos os IDs já haviam sido processados'
        )
        del bloco, representantes, representantes_novos
        gc.collect()
        continue

    geometria = serie_wkt_para_geometria(
        representantes_novos[COLUNA_GEOMETRIA_SICOR]
    )

    gdf_ids = gpd.GeoDataFrame(
        representantes_novos[
            [COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
        ].copy(),
        geometry=geometria,
        crs=CRS_SICOR,
    )

    gdf_ids, invalidas_corrigidas, descartadas = corrigir_geometrias(gdf_ids)
    gdf_ids = gdf_ids.reset_index(drop=True)

    total_geometrias_invalidas_corrigidas += invalidas_corrigidas
    total_geometrias_descartadas += descartadas
    total_ids_geometria_valida += len(gdf_ids)
    ids_com_geometria_valida.update(
        gdf_ids[COLUNA_ID_GLEBA2].dropna().astype(str).tolist()
    )

    if gdf_ids.empty:
        print(
            f'Bloco {numero_bloco:03d}: IDs novos={len(representantes_novos):,} | '
            'nenhuma geometria válida'
        )
        del bloco, representantes, representantes_novos, gdf_ids
        gc.collect()
        continue

    candidatos = gpd.sjoin(
        gdf_ids,
        embargos,
        how='inner',
        predicate='intersects',
    ).reset_index(drop=True)

    total_candidatos_espaciais += len(candidatos)

    if candidatos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            f'IDs novos={len(representantes_novos):,} | interseções=0 | '
            f'repetições puladas={len(bloco) - len(representantes_novos):,}'
        )
        del bloco, representantes, representantes_novos, gdf_ids, candidatos
        gc.collect()
        continue

    candidatos['GEOMETRIA_EMBARGO'] = (
        candidatos['index_right'].map(embargos.geometry)
    )

    candidatos = candidatos.dropna(
        subset=['GEOMETRIA_EMBARGO']
    ).reset_index(drop=True)

    geometria_gleba_wgs84 = gpd.GeoSeries(
        candidatos.geometry.to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_embargo_wgs84 = gpd.GeoSeries(
        candidatos['GEOMETRIA_EMBARGO'].to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_gleba_area = geometria_gleba_wgs84.to_crs(CRS_AREA)
    geometria_embargo_area = geometria_embargo_wgs84.to_crs(CRS_AREA)

    geometria_gleba_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_gleba_area],
        index=pd.RangeIndex(len(geometria_gleba_area)),
        crs=CRS_AREA,
    )

    geometria_embargo_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_embargo_area],
        index=pd.RangeIndex(len(geometria_embargo_area)),
        crs=CRS_AREA,
    )

    (
        geometria_intersecao_area,
        recuperadas_bloco,
        falhas_bloco,
    ) = calcular_intersecoes_seguras(
        geometria_gleba_area,
        geometria_embargo_area,
        CRS_AREA,
    )

    total_intersecoes_recuperadas += recuperadas_bloco
    total_falhas_topologicas += falhas_bloco

    candidatos['AREA_GLEBA_HA_CALCULADA'] = (
        geometria_gleba_area.area.to_numpy() / 10_000
    )
    candidatos['AREA_INTERSECAO_HA'] = (
        geometria_intersecao_area.area.to_numpy() / 10_000
    )
    candidatos['PERCENTUAL_GLEBA_INTERSECTADA'] = np.where(
        candidatos['AREA_GLEBA_HA_CALCULADA'] > 0,
        candidatos['AREA_INTERSECAO_HA']
        / candidatos['AREA_GLEBA_HA_CALCULADA']
        * 100,
        np.nan,
    )

    mascara_area_positiva = (
        candidatos['AREA_INTERSECAO_HA'].fillna(0).gt(0).to_numpy()
    )
    total_contatos_sem_area += int((~mascara_area_positiva).sum())

    if not mascara_area_positiva.any():
        print(
            f'Bloco {numero_bloco:03d}: candidatos={len(candidatos):,} | '
            'nenhuma interseção com área positiva'
        )
        del (
            bloco, representantes, representantes_novos, gdf_ids, candidatos,
            geometria_gleba_wgs84, geometria_embargo_wgs84,
            geometria_gleba_area, geometria_embargo_area,
            geometria_intersecao_area, mascara_area_positiva,
        )
        gc.collect()
        continue

    candidatos = candidatos.loc[mascara_area_positiva].reset_index(drop=True)

    geometria_intersecao_area_valida = gpd.GeoSeries(
        geometria_intersecao_area.to_numpy()[mascara_area_positiva],
        index=pd.RangeIndex(int(mascara_area_positiva.sum())),
        crs=CRS_AREA,
    )
    geometria_intersecao_wgs84 = geometria_intersecao_area_valida.to_crs(CRS_SICOR)

    candidatos['GEOMETRIA_EMBARGO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in candidatos['GEOMETRIA_EMBARGO']
    ]
    candidatos['GEOMETRIA_INTERSECAO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in geometria_intersecao_wgs84
    ]
    candidatos['TIPO_IMPEDIMENTO'] = 'Embargo Ambiental - IBAMA'

    colunas_resultado = [
        COLUNA_ID_GLEBA2,
        *COLUNAS_EMB_SAIDA,
        'AREA_GLEBA_HA_CALCULADA',
        'AREA_INTERSECAO_HA',
        'PERCENTUAL_GLEBA_INTERSECTADA',
        'GEOMETRIA_EMBARGO_WKT',
        'GEOMETRIA_INTERSECAO_WKT',
        'TIPO_IMPEDIMENTO',
    ]

    resultado_bloco = pd.DataFrame(candidatos[colunas_resultado])
    resultados_intersecao.append(resultado_bloco)
    total_intersecoes_validas += len(resultado_bloco)

    print(
        f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
        f'IDs novos={len(representantes_novos):,} | '
        f'repetições puladas={len(bloco) - len(representantes_novos):,} | '
        f'candidatos={len(mascara_area_positiva):,} | '
        f'interseções com área={len(resultado_bloco):,}'
    )

    del (
        bloco, representantes, representantes_novos, gdf_ids, candidatos,
        geometria_gleba_wgs84, geometria_embargo_wgs84,
        geometria_gleba_area, geometria_embargo_area,
        geometria_intersecao_area, geometria_intersecao_area_valida,
        geometria_intersecao_wgs84, mascara_area_positiva, resultado_bloco,
    )
    gc.collect()

print('\nPROCESSAMENTO ESPACIAL CONCLUÍDO')
print('=' * 90)
print('Linhas lidas:', f'{total_linhas_lidas:,}')
print('Linhas com ID e geometria:', f'{total_linhas_com_id_e_geometria:,}')
print('ID_GLEBA2 distintos encontrados:', f'{len(todos_ids_encontrados):,}')
print('ID_GLEBA2 distintos com WKT:', f'{len(assinatura_geometria_por_id):,}')
print('IDs submetidos ao teste:', f'{total_ids_novos:,}')
print('Repetições que não foram testadas novamente:', f'{total_repeticoes_puladas:,}')
print('IDs com geometria válida:', f'{total_ids_geometria_valida:,}')
print('Geometrias inválidas corrigidas:', f'{total_geometrias_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_geometrias_descartadas:,}')
print('Candidatos espaciais:', f'{total_candidatos_espaciais:,}')
print('Interseções recuperadas:', f'{total_intersecoes_recuperadas:,}')
print('Falhas topológicas:', f'{total_falhas_topologicas:,}')
print('Contatos sem área positiva:', f'{total_contatos_sem_area:,}')
print('Interseções válidas ID_GLEBA2 × embargo:', f'{total_intersecoes_validas:,}')


Bloco 001: lidas=20,000 | IDs novos=18,663 | repetições puladas=1,337 | candidatos=74 | interseções com área=73
Bloco 002: lidas=20,000 | IDs novos=17,690 | repetições puladas=2,310 | candidatos=106 | interseções com área=103
Bloco 003: lidas=20,000 | IDs novos=17,256 | repetições puladas=2,744 | candidatos=69 | interseções com área=69
Bloco 004: lidas=20,000 | IDs novos=17,061 | repetições puladas=2,939 | candidatos=50 | interseções com área=49
Bloco 005: lidas=20,000 | IDs novos=16,706 | repetições puladas=3,294 | candidatos=59 | interseções com área=59
Bloco 006: lidas=20,000 | IDs novos=14,364 | repetições puladas=5,636 | candidatos=48 | interseções com área=48
Bloco 007: lidas=20,000 | IDs novos=15,135 | repetições puladas=4,865 | candidatos=40 | interseções com área=35
Bloco 008: lidas=20,000 | IDs novos=15,230 | repetições puladas=4,770 | candidatos=36 | interseções com área=36
Bloco 009: lidas=20,000 | IDs novos=14,954 | repetições puladas=5,046 | candidatos=29 | interseções co

In [ ]:
# ============================================================
# 7. CONSOLIDAR E SALVAR O RESULTADO ÚNICO POR ID_GLEBA2
# ============================================================

if resultados_intersecao:
    intersecoes_por_id = pd.concat(
        resultados_intersecao,
        ignore_index=True,
        sort=False,
    )
else:
    intersecoes_por_id = pd.DataFrame(
        columns=[COLUNA_ID_GLEBA2, 'EMB_ID_INTERNO']
    )

del resultados_intersecao
gc.collect()

quantidade_antes = len(intersecoes_por_id)

if not intersecoes_por_id.empty:
    intersecoes_por_id = (
        intersecoes_por_id
        .sort_values('AREA_INTERSECAO_HA', ascending=False)
        .drop_duplicates(
            subset=[COLUNA_ID_GLEBA2, 'EMB_ID_INTERNO'],
            keep='first',
        )
        .reset_index(drop=True)
    )

print(
    'Duplicações técnicas ID_GLEBA2 × embargo removidas:',
    f'{quantidade_antes - len(intersecoes_por_id):,}',
)

intersecoes_por_id.to_csv(
    ARQUIVO_INTERSECAO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

# O resumo inclui todos os IDs, inclusive os sem interseção.
todos_ids = pd.DataFrame({
    COLUNA_ID_GLEBA2: sorted(todos_ids_encontrados)
})

todos_ids['TEM_GEOMETRIA_VALIDA'] = (
    todos_ids[COLUNA_ID_GLEBA2].isin(ids_com_geometria_valida)
)

if intersecoes_por_id.empty:
    resumo_por_id = todos_ids.copy()
    resumo_por_id['TEM_INTERSECAO_EMBARGO'] = False
    resumo_por_id['QUANTIDADE_EMBARGOS_INTERSECTADOS'] = 0
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = 0.0
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = 0.0
else:
    resumo_positivo = (
        intersecoes_por_id.groupby(COLUNA_ID_GLEBA2, dropna=False)
        .agg(
            QUANTIDADE_EMBARGOS_INTERSECTADOS=(
                'EMB_ID_INTERNO', 'nunique'
            ),
            AREA_INTERSECAO_TOTAL_HA=('AREA_INTERSECAO_HA', 'sum'),
            PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA=(
                'PERCENTUAL_GLEBA_INTERSECTADA', 'max'
            ),
        )
        .reset_index()
    )

    resumo_por_id = todos_ids.merge(
        resumo_positivo,
        on=COLUNA_ID_GLEBA2,
        how='left',
        validate='one_to_one',
    )

    resumo_por_id['TEM_INTERSECAO_EMBARGO'] = (
        resumo_por_id['QUANTIDADE_EMBARGOS_INTERSECTADOS']
        .notna()
    )
    resumo_por_id['QUANTIDADE_EMBARGOS_INTERSECTADOS'] = (
        resumo_por_id['QUANTIDADE_EMBARGOS_INTERSECTADOS']
        .fillna(0)
        .astype('int64')
    )
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = (
        resumo_por_id['AREA_INTERSECAO_TOTAL_HA'].fillna(0.0)
    )
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = (
        resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'].fillna(0.0)
    )

    resumo_por_id = resumo_por_id[
        [
            COLUNA_ID_GLEBA2,
            'TEM_GEOMETRIA_VALIDA',
            'TEM_INTERSECAO_EMBARGO',
            'QUANTIDADE_EMBARGOS_INTERSECTADOS',
            'AREA_INTERSECAO_TOTAL_HA',
            'PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA',
        ]
    ]

resumo_por_id.to_csv(
    ARQUIVO_RESUMO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

print('\nResultado único salvo em:')
print(ARQUIVO_INTERSECAO_POR_ID)
print('\nResumo de todos os IDs salvo em:')
print(ARQUIVO_RESUMO_POR_ID)
print('\nIDs com interseção:', f"{resumo_por_id['TEM_INTERSECAO_EMBARGO'].sum():,}")
print('IDs sem geometria válida:', f"{(~resumo_por_id['TEM_GEOMETRIA_VALIDA']).sum():,}")
print('IDs sem interseção:', f"{(~resumo_por_id['TEM_INTERSECAO_EMBARGO']).sum():,}")


Duplicações técnicas ID_GLEBA2 × embargo removidas: 0

Resultado único salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/embargos/resultados_por_id_gleba2/intersecao_embargos_unica_por_id_gleba2.csv

Resumo de todos os IDs salvo em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/embargos/resultados_por_id_gleba2/resumo_embargos_por_id_gleba2.csv

IDs com interseção: 626
IDs sem geometria válida: 0
IDs sem interseção: 394,949


In [ ]:
# ============================================================
# 8. DEVOLVER A RESPOSTA A TODAS AS LINHAS DO SICOR
# ============================================================

# A interseção não é recalculada aqui. O merge apenas replica a resposta
# de cada ID_GLEBA2 para todos os REF_BACEN, ordens, índices e anos ligados a ele.

colunas_espaciais_para_merge = [
    coluna
    for coluna in intersecoes_por_id.columns
    if coluna != COLUNA_ID_GLEBA2
]

escrever_cabecalho = True
total_linhas_originais_reprocessadas = 0
total_linhas_expandidas = 0

leitor_expandir = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_expandir, start=1):
    bloco = normalizar_colunas(bloco)
    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    total_linhas_originais_reprocessadas += len(bloco)

    if intersecoes_por_id.empty:
        expandido = pd.DataFrame()
    else:
        expandido = bloco.merge(
            intersecoes_por_id,
            on=COLUNA_ID_GLEBA2,
            how='inner',
            validate='many_to_many',
            suffixes=('', '_INTERSECAO'),
        )

    if expandido.empty:
        print(f'Bloco {numero_bloco:03d}: nenhuma linha com interseção')
        del bloco, expandido
        gc.collect()
        continue

    expandido.to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        mode='w' if escrever_cabecalho else 'a',
        header=escrever_cabecalho,
        index=False,
        sep=SEPARADOR_SAIDA,
        decimal=',',
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
    )

    total_linhas_expandidas += len(expandido)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: originais={len(bloco):,} | '
        f'linhas de interseção expandidas={len(expandido):,}'
    )

    del bloco, expandido
    gc.collect()

if escrever_cabecalho:
    # Nenhuma interseção: ainda cria um CSV vazio com cabeçalho conhecido.
    pd.DataFrame(
        columns=[*colunas_sicor, *colunas_espaciais_para_merge]
    ).to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        index=False,
        sep=SEPARADOR_SAIDA,
        encoding='utf-8-sig',
    )

if total_linhas_originais_reprocessadas != total_linhas_lidas:
    raise RuntimeError(
        'A quantidade de linhas relidas para a expansão difere da '
        'quantidade lida na etapa espacial.'
    )

print('\nRESULTADO DETALHADO EXPANDIDO')
print('=' * 90)
print('Linhas originais relidas:', f'{total_linhas_originais_reprocessadas:,}')
print('Linhas finais de interseção:', f'{total_linhas_expandidas:,}')
print('Arquivo salvo em:')
print(ARQUIVO_INTERSECAO_EXPANDIDA)


Bloco 001: originais=20,000 | linhas de interseção expandidas=80
Bloco 002: originais=20,000 | linhas de interseção expandidas=132
Bloco 003: originais=20,000 | linhas de interseção expandidas=81
Bloco 004: originais=20,000 | linhas de interseção expandidas=74
Bloco 005: originais=20,000 | linhas de interseção expandidas=71
Bloco 006: originais=20,000 | linhas de interseção expandidas=68
Bloco 007: originais=20,000 | linhas de interseção expandidas=62
Bloco 008: originais=20,000 | linhas de interseção expandidas=44
Bloco 009: originais=20,000 | linhas de interseção expandidas=37
Bloco 010: originais=20,000 | linhas de interseção expandidas=36
Bloco 011: originais=20,000 | linhas de interseção expandidas=44
Bloco 012: originais=20,000 | linhas de interseção expandidas=30
Bloco 013: originais=20,000 | linhas de interseção expandidas=21
Bloco 014: originais=20,000 | linhas de interseção expandidas=34
Bloco 015: originais=20,000 | linhas de interseção expandidas=16
Bloco 016: originais=20,

In [ ]:
# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print('ARQUIVOS GERADOS')
print('=' * 100)

for caminho in [
    ARQUIVO_INTERSECAO_POR_ID,
    ARQUIVO_INTERSECAO_EXPANDIDA,
    ARQUIVO_RESUMO_POR_ID,
]:
    print(f"{'OK' if caminho.exists() else 'NÃO CRIADO'} | {caminho}")

print('\nAMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2')
display(intersecoes_por_id.head(5))

print('\nAMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO')
display(resumo_por_id.head(5))

print('\nESTATÍSTICAS')
print('ID_GLEBA2 distintos analisados:', f'{len(resumo_por_id):,}')
print('ID_GLEBA2 com interseção:', f"{resumo_por_id['TEM_INTERSECAO_EMBARGO'].sum():,}")
print('Pares ID_GLEBA2 × feição de embargo:', f'{len(intersecoes_por_id):,}')

if not intersecoes_por_id.empty:
    print(
        'Área total das interseções únicas (ha):',
        f"{intersecoes_por_id['AREA_INTERSECAO_HA'].sum():,.4f}",
    )


ARQUIVOS GERADOS
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/embargos/resultados_por_id_gleba2/intersecao_embargos_unica_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/embargos/resultados_por_id_gleba2/intersecao_sicor_embargos_expandida_por_id_gleba2.csv
OK | /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Arquivos gerados e relatorios/intersecoes/embargos/resultados_por_id_gleba2/resumo_embargos_por_id_gleba2.csv

AMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2


,ID_GLEBA2,EMB_OBJECTID,EMB_SEQ_TAD,EMB_NUM_TAD,EMB_SERIE_TAD,EMB_OPERACAO,EMB_ORIGEM_GEO,EMB_COD_UF,EMB_UF,EMB_COD_MUNICI,EMB_MUNICIPIO,EMB_NOME_IMOVE,EMB_DES_LOCALI,EMB_NOME_EMBAR,EMB_CPF_CNPJ_E,EMB_SIT_DESMAT,EMB_TIPO_AREA,EMB_NUM_AUTO_I,EMB_SERIE_AUTO,EMB_COD_TIPO_B,EMB_DES_TIPO_B,EMB_UNID_CONTR,EMB_ORDEM_FISC,EMB_CD_ACAO_FI,EMB_NUM_PROCES,EMB_DES_TAD,EMB_DES_INFRAC,EMB_NUM_LONGIT,EMB_NUM_LATITU,EMB_DAT_EMBARG,EMB_DAT_IMPRES,EMB_DAT_ULT_AL,EMB_NUM_LONG00,EMB_NUM_LATI00,EMB_QTD_AREA_D,EMB_QTD_AREA_E,EMB_DAT_ULT_00,EMB_ST_AREA_SH,EMB_ST_PERIMET,EMB_ID_INTERNO,EMB_AREA_TOTAL_HA,AREA_GLEBA_HA_CALCULADA,AREA_INTERSECAO_HA,PERCENTUAL_GLEBA_INTERSECTADA,GEOMETRIA_EMBARGO_WKT,GEOMETRIA_INTERSECAO_WKT,TIPO_IMPEDIMENTO
0,173340,6037,513642,525650,C,None,Polígono,29,BA,2911105,Formosa do Rio Preto,"Fazenda Dalva, Coaceral.","Fazenda Dalva, Coaceral.",AUGUSTO MUNBACH,62368397000,D,None,475668,D,None,None,DIFIS - Salvador/BA,None,None,02006002079200844,"Fica embargada a área de 2288 hectares de vegetação nativa de cerrado desmatada a corte raso sem autorização dos órgãos ambientais, sendo 1300 hectares no interior do Parque Na...","Explorar ou danificar floresta ou qualquer tipo de vegetação nativa ou de espécies nativas plantadas, localizada fora de área de reserva legal averbada, de domínio público ou p...",45° 35' 43.001'' W,10° 10' 32.002'' S,14/11/08 08:56:00,01/01/01 00:00:00,07/05/24 10:48:54,-45.595280,-10.175556,2288.0,2241.083008,07/05/24 10:48:54,0.001849,0.263641,55472,2264.755857,638.191966,618.656189,96.938887,"POLYGON ((-45.59292199999993 -10.245416999999975, -45.595357999999976 -10.175643999999977, -45.57649199999997 -10.175482999999929, -45.57380299999994 -10.18324599999994, -45.57...","POLYGON ((-45.573931002829866 -10.183562999984387, -45.5765589737149 -10.202893010085841, -45.59449527039103 -10.200399001639713, -45.59534184541011 -10.176109518405035, -45.57...",Embargo Ambiental - IBAMA
1,78717,6037,513642,525650,C,None,Polígono,29,BA,2911105,Formosa do Rio Preto,"Fazenda Dalva, Coaceral.","Fazenda Dalva, Coaceral.",AUGUSTO MUNBACH,62368397000,D,None,475668,D,None,None,DIFIS - Salvador/BA,None,None,02006002079200844,"Fica embargada a área de 2288 hectares de vegetação nativa de cerrado desmatada a corte raso sem autorização dos órgãos ambientais, sendo 1300 hectares no interior do Parque Na...","Explorar ou danificar floresta ou qualquer tipo de vegetação nativa ou de espécies nativas plantadas, localizada fora de área de reserva legal averbada, de domínio público ou p...",45° 35' 43.001'' W,10° 10' 32.002'' S,14/11/08 08:56:00,01/01/01 00:00:00,07/05/24 10:48:54,-45.595280,-10.175556,2288.0,2241.083008,07/05/24 10:48:54,0.001849,0.263641,55472,2264.755857,636.420687,617.210743,96.981565,"POLYGON ((-45.59292199999993 -10.245416999999975, -45.595357999999976 -10.175643999999977, -45.57649199999997 -10.175482999999929, -45.57380299999994 -10.18324599999994, -45.57...","POLYGON ((-45.573932006350745 -10.183563025569587, -45.576276112539055 -10.2013009931948, -45.57648735374919 -10.202842942992913, -45.59450011132444 -10.200260668711996, -45.59...",Embargo Ambiental - IBAMA
2,216425,6037,513642,525650,C,None,Polígono,29,BA,2911105,Formosa do Rio Preto,"Fazenda Dalva, Coaceral.","Fazenda Dalva, Coaceral.",AUGUSTO MUNBACH,62368397000,D,None,475668,D,None,None,DIFIS - Salvador/BA,None,None,02006002079200844,"Fica embargada a área de 2288 hectares de vegetação nativa de cerrado desmatada a corte raso sem autorização dos órgãos ambientais, sendo 1300 hectares no interior do Parque Na...","Explorar ou danificar floresta ou qualquer tipo de vegetação nativa ou de espécies nativas plantadas, localizada fora de área de reserva legal averbada, de domínio público ou p...",45° 35' 43.001'' W,10° 10' 32.002'' S,14/11/08 08:56:00,01/01/01 00:00:00,07/05/24 10:48:54,-45.595280,-10.175556,2288.0,2241.083008,07/05/24 10:48:54,0.001849,0.263641,55472,2264.755857,636.407853,617.209892,96.983387,"POLYGON ((-45.59292199999993 -10.2454169999


AMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO


,ID_GLEBA2,TEM_GEOMETRIA_VALIDA,TEM_INTERSECAO_EMBARGO,QUANTIDADE_EMBARGOS_INTERSECTADOS,AREA_INTERSECAO_TOTAL_HA,PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA
0,1,True,False,0,0.0,0.0
1,10,True,False,0,0.0,0.0
2,100,True,False,0,0.0,0.0
3,1000,True,False,0,0.0,0.0
4,10000,True,False,0,0.0,0.0



ESTATÍSTICAS
ID_GLEBA2 distintos analisados: 395,575
ID_GLEBA2 com interseção: 626
Pares ID_GLEBA2 × feição de embargo: 725
Área total das interseções únicas (ha): 20,661.8427
